# **Use this step-by-step notebook to update the resources available on the Knowledge Archive webpage**

> Upload the updated excel spreadsheet below




In [1]:
#The upload file button will appear after this cell is run

import pandas as pd
import re
import os

from google.colab import files
uploaded=files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  filename = fn

Saving Knowledge Archives Data.xlsx to Knowledge Archives Data.xlsx
User uploaded file "Knowledge Archives Data.xlsx" with length 694832 bytes


In [2]:
ifilename = next(iter(uploaded.keys())) # Get the first uploaded filename
else:
    raise FileNotFoundError("No file has been uploaded.")

# Create a new dataframe (basic spreadsheet style) from all of the separate sheets in this spreadsheet
all_sheets = pd.read_excel(filename, sheet_name=None, header=None) # Read without a header initially
dfs_with_journal = []
for sheet_name, sheet_df in all_sheets.items():
    if sheet_df.empty or sheet_df.shape[0] < 3:
        print(f"Skipping empty or short sheet: {sheet_name}")
        continue

    # Get the first row for the journal name
    journal_name_series = sheet_df.iloc[0]
    journal_name = ' '.join(journal_name_series.dropna().astype(str)).strip()
    # Further clean up: remove newline characters and extra whitespace
    journal_name = journal_name.replace('\n', '').strip()


    # Use the third row as the header and clean up column names
    header_row = sheet_df.iloc[2]
    header_row = header_row.astype(str).str.replace('nan', '').str.strip()


    sheet_df = sheet_df[3:].copy() # Data starts from the 4th row

    # Handle duplicate column names in the header
    cols = pd.Series(header_row)
    for dup in cols[cols.duplicated()].unique():
        cols[cols[cols == dup].index.values.tolist()] = [dup + '.' + str(i) if i != 0 else dup for i in range(len(cols[cols == dup].index.values.tolist()))]
    sheet_df.columns = cols

    # Print column names and their types for debugging
    print(f"Sheet: {sheet_name}")
    print("Columns and dtypes before concatenation:")
    print(sheet_df.columns)
    print(sheet_df.dtypes)
    print("-" * 30)


    # Add the journal name as a new column
    sheet_df['journal'] = journal_name
    dfs_with_journal.append(sheet_df)

# Concatenate all dataframes with journal information
df = pd.concat(dfs_with_journal, ignore_index=True)

# Select only the desired columns and exclude those starting with '.'
desired_columns = ['title', 'filename', 'file_link', 'year', 'edition_or_version', 'author', 'tags', 'journal', 'image']
# Ensure all desired columns exist before selecting
desired_columns_existing = [col for col in desired_columns if col in df.columns]
df = df[desired_columns_existing]


# Fill missing values with empty strings
df = df.fillna('')


# Rename the 'edition_or_version' column to 'volume' if it exists
if 'edition_or_version' in df.columns:
    df = df.rename(columns={'edition_or_version': 'volume'})


# Function to extract year, handling errors and various formats, including multiple years
def extract_year(date_str):
    if pd.isna(date_str) or date_str == '':
        return []  # Return empty list for missing or empty values
    try:
        # Convert to string and find all occurrences of four-digit numbers
        date_str = str(date_str).strip()
        year_matches = re.findall(r'\d{4}', date_str)

        extracted_years = []
        for year_str in year_matches:
            try:
                extracted_years.append(year_str) # Keep as string
            except ValueError:
                # Handle cases where a four-digit string is not a valid integer (shouldn't happen with \d{4} but as a safeguard)
                print(f"Warning: Could not convert '{year_str}' to integer in year string '{date_str}'")
                pass # Skip this invalid year

        return extracted_years if extracted_years else [] # Return the list of years or an empty list if none found

    except:
        # If any other error occurs, return an empty list
        return []


# Apply the function to the 'year' column
if 'year' in df.columns:
    df['year'] = df['year'].apply(extract_year)


# Define the columns to check for emptiness, excluding 'journal' and 'year'
columns_to_check_for_meaningful_text = [col for col in df.columns if col not in ['journal', 'year']]

# Remove rows where all specified columns (excluding 'journal' and 'year') are empty strings
df = df[~df[columns_to_check_for_meaningful_text].eq('').all(axis=1)]


df

Sheet: Aghamtao
Columns and dtypes before concatenation:
Index(['title', 'filename', 'file_link', 'year', 'edition_or_version',
       'author', 'tags'],
      dtype='object', name=2)
2
title                 object
filename              object
file_link             object
year                  object
edition_or_version    object
author                object
tags                  object
dtype: object
------------------------------
Sheet: Historical Bulletin
Columns and dtypes before concatenation:
Index(['title', 'filename', 'file_link', 'year', 'edition_or_version',
       'author', 'tags', '', '.1', '.2', '.3'],
      dtype='object', name=2)
2
title                  object
filename               object
file_link              object
year                   object
edition_or_version     object
author                 object
tags                   object
                      float64
.1                    float64
.2                    float64
.3                     object
dtype: object
---

2,title,filename,file_link,year,volume,author,tags,journal
0,Selected of the First National Conference of UGAT,01 Title Page,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],Vol 1,,"Title, Aghamtao",Aghamtao
1,Publisher,02 Publisher,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],Vol 1,,Publisher,Aghamtao
2,Editor,03 Editor,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],Vol 1,"Montepio, Susan N.",Editor,Aghamtao
3,Foreword,04 Foreword,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],Vol 1,,Foreword,Aghamtao
4,Editor’s Note,05 Editor’s Note,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],Vol 1,,Editor'S Note,Aghamtao
...,...,...,...,...,...,...,...,...
4567,Approaches in Forecasting Cereals Production,14 Approaches in Forecasting Cereals Production,https://pssc.org.ph/wp-content/pssc-archives/T...,[2007],"VOL 56, NUM 3 - 4","dela Paz-Nalica, Angela, Barrios, Erniel B.",FORECASTING CEREALS PRODUCTION,The Philippine Statistician
4568,References,15 References,https://pssc.org.ph/wp-content/pssc-archives/T...,[2007],"VOL 56, NUM 3 - 4",,"REFERENCES, APPENDIX",The Philippine Statistician
4570,The Philippine Statistician Conference Volume 57,"Vol L V I I, 2008",https://pssc.org.ph/wp-content/pssc-archives/T...,[2008],VOL 57,The Philippine Statistician,"OVERSEAS FILIPINOS', POPULATION DYNAMICS, HOUS...",The Philippine Statistician
4572,The Philippine Statistician Conference Volume 59,T P S 2010 ( F U L L),https://pssc.org.ph/wp-content/pssc-archives/T...,[2010],VOL 59,The Philippine Statistician,"DYNAMIC CONDITIONAL CORRELATION, ROREIGN EXCHA...",The Philippine Statistician


In [3]:
#Function to convert Roman numerals to integers
def roman_to_int(s):
    """Converts a Roman numeral string to an integer.
    Handles standard Roman numerals up to M (1000).
    """
    roman_map = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    integer_val = 0
    i = 0
    # Convert input to uppercase for case-insensitive matching
    s = s.upper()
    while i < len(s):
        # Check for subtractive cases (e.g., IV, IX, XL, XC, CD, CM)
        if i + 1 < len(s) and s[i] in roman_map and s[i+1] in roman_map and roman_map[s[i]] < roman_map[s[i+1]]:
            integer_val += roman_map[s[i+1]] - roman_map[s[i]]
            i += 2
        # Otherwise, add the value of the current Roman numeral
        elif s[i] in roman_map:
            integer_val += roman_map[s[i]]
            i += 1
        else:
            # If an invalid character is encountered, return None or handle as error
            print(f"Warning: Invalid character '{s[i]}' in Roman numeral string '{s}'")
            return None # Return None for invalid input
    return integer_val

#Normalization function for the 'edition_or_version' column
def normalize_edition(edition_str):
    """
    Normalizes edition/version strings according to the rules:
    1. Converts Roman numerals to Arabic.
    2. Extracts all numbers (Arabic or Roman) that appear to be volume numbers.
    """

    # Convert to string to handle potential non-string inputs and strip whitespace
    edition_str = str(edition_str).strip()
    edition_str_upper = edition_str.upper()

    # Find all sequences of Arabic digits or Roman numerals (I, V, X, L)
    # that are preceded by "VOL", "V.", "VOLUME", or appear as standalone numbers.
    # Exclude single 'L' unless it's part of a larger Roman numeral or a standalone 'L' volume.
    # This revised pattern looks for:
    # 1. "VOL", "V.", or "VOLUME" followed by a number (Arabic or Roman)
    # 2. Standalone Arabic numbers
    # 3. Standalone Roman numerals (more than one character or a single character that is not 'L')
    # 4. A standalone 'L' if it's the only character or part of a larger Roman numeral match
    number_matches = re.findall(r'(?:VOL|V\.|VOLUME)\s*([IVXL]+\b|\d+)|(\d+)|([IVXL]{2,}\b|[IVX]\b)', edition_str_upper)


    normalized_volumes = []
    for match in number_matches:
        # The match will be a tuple because of the OR conditions in the regex.
        # We need to find the non-empty group.
        number_part = ''.join(match)

        if not number_part:
            continue # Skip empty matches


        # Check if the extracted part is a Roman numeral
        if re.fullmatch(r'^[IVXL]+$', number_part):
             # Verify all characters are valid Roman numerals before attempting conversion
            if all(char in 'IVXL' for char in number_part):
                converted_num = roman_to_int(number_part)
                if converted_num is not None:
                    normalized_volumes.append(str(converted_num))
                else:
                    # Handle cases where roman_to_int returned None due to invalid characters
                    print(f"Warning: Could not convert Roman numeral '{number_part}' in string '{edition_str}'")
            else:
                 print(f"Warning: Invalid characters in potential Roman numeral '{number_part}' in string '{edition_str}'")


        # Check if the extracted part is an Arabic numeral
        elif re.fullmatch(r'^\d+$', number_part):
            normalized_volumes.append(str(int(number_part))) # Convert to int then str to remove leading zeros (e.g., "01" -> "1")

    return normalized_volumes if normalized_volumes else [] # Return list of volumes or empty list


# Apply the normalization function to your 'edition_or_version' column
df['volume'] = df['volume'].apply(normalize_edition)

df

2,title,filename,file_link,year,volume,author,tags,journal
0,Selected of the First National Conference of UGAT,01 Title Page,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],[1],,"Title, Aghamtao",Aghamtao
1,Publisher,02 Publisher,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],[1],,Publisher,Aghamtao
2,Editor,03 Editor,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],[1],"Montepio, Susan N.",Editor,Aghamtao
3,Foreword,04 Foreword,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],[1],,Foreword,Aghamtao
4,Editor’s Note,05 Editor’s Note,https://pssc.org.ph/wp-content/pssc-archives/A...,[1978],[1],,Editor'S Note,Aghamtao
...,...,...,...,...,...,...,...,...
4567,Approaches in Forecasting Cereals Production,14 Approaches in Forecasting Cereals Production,https://pssc.org.ph/wp-content/pssc-archives/T...,[2007],"[56, 3, 4]","dela Paz-Nalica, Angela, Barrios, Erniel B.",FORECASTING CEREALS PRODUCTION,The Philippine Statistician
4568,References,15 References,https://pssc.org.ph/wp-content/pssc-archives/T...,[2007],"[56, 3, 4]",,"REFERENCES, APPENDIX",The Philippine Statistician
4570,The Philippine Statistician Conference Volume 57,"Vol L V I I, 2008",https://pssc.org.ph/wp-content/pssc-archives/T...,[2008],[57],The Philippine Statistician,"OVERSEAS FILIPINOS', POPULATION DYNAMICS, HOUS...",The Philippine Statistician
4572,The Philippine Statistician Conference Volume 59,T P S 2010 ( F U L L),https://pssc.org.ph/wp-content/pssc-archives/T...,[2010],[59],The Philippine Statistician,"DYNAMIC CONDITIONAL CORRELATION, ROREIGN EXCHA...",The Philippine Statistician


In [4]:
# Function to extract year, handling errors and various formats, including multiple years
def extract_year(date_str):
    # Check if the input is a list and if it's empty or contains only None/empty strings
    if isinstance(date_str, list):
        if not date_str or all(pd.isna(x) or x == '' for x in date_str):
            return [] # Return empty list for empty or all-empty lists
    # Handle non-list inputs (like individual NaNs or empty strings that might still come through)
    elif pd.isna(date_str) or date_str == '':
        return []  # Return empty list for missing or empty values

    try:
        # Convert to string and handle potential year ranges like "1982 & 1983" or "1982 and 1983"
        date_str = str(date_str).strip()
        # Find all occurrences of four-digit numbers
        year_matches = re.findall(r'\d{4}', date_str)

        extracted_years = []
        for year_str in year_matches:
            try:
                # Append the year as a string
                extracted_years.append(year_str)
            except ValueError:
                # Handle cases where a four-digit string is not a valid integer (shouldn't happen with \d{4} but as a safeguard)
                print(f"Warning: Could not convert '{year_str}' to integer in year string '{date_str}'")
                pass # Skip this invalid year

        return extracted_years if extracted_years else [] # Return the list of years or an empty list if none found

    except:
        # If any other error occurs, return an empty list
        return []

# Apply the function to the 'year' column
if 'year' in df.columns:
    # Ensure the column is treated as objects before applying the function that returns lists
    df['year'] = df['year'].astype(object).apply(extract_year)

**These cells convert the pdfs to images and handle any issues**

In [5]:
# --- Convert DataFrame to JSON ---
json_output_string = df.to_json(orient='records', indent=2)

In [11]:
# --- Save JSON File ---
output_json_dir = "output_json"
output_json_path = os.path.join(output_json_dir, "KAdata.json")  # ← Change filename
os.makedirs(output_json_dir, exist_ok=True)

with open(output_json_path, 'w', encoding='utf-8') as f:
    f.write(json_output_string)

print(f"\nJSON file successfully created at: {output_json_path}")


JSON file successfully created at: output_json/KAdata.json


## Run these last two cells to download the new .json file and folder with images

In [15]:

from google.colab import files
files.download('output_json/PCRNdata.json')  # ← Use this to change filename

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>